<a href="https://colab.research.google.com/github/hyeonji-jang65/econ81230-project-jang/blob/main/code/01_sample_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pyreadstat

Mounted at /content/drive


In [ ]:
import numpy as np, pandas as pd, pyreadstat

PATH = '/content/drive/MyDrive/HRS/randhrs1992_2022v1.dta'

_, meta = pyreadstat.read_dta(PATH, metadataonly=True)
lower = {c.lower(): c for c in meta.column_names}

W = range(1, 17)
fixed = ['hhidpn', 'hacohort', 'rabyear', 'ragender', 'raedyrs', 'ravetrn']
stems = ['inw', 'peninc', 'peni_n', 'work', 'slfemp', 'samemp', 'jcpen', 'penct',
         'ptyp1', 'ptyp2', 'ptyp3', 'ptyp4',
         'dcbal1', 'dcbal2', 'dcbal3', 'dcbal4',
         'iearn', 'ipen', 'iirawy1', 'isret', 'jcten', 'jcocc',
         'agey_e', 'mstat', 'cendiv', 'wtresp', 'iwendy', 'ifpen']

def vname(stem, w):
    return f'inw{w}' if stem == 'inw' else f'r{w}{stem}'

cols = [lower[v] for v in fixed if v in lower]
cols += [lower[vname(s, w)] for s in stems for w in W if vname(s, w) in lower]

df, meta2 = pyreadstat.read_dta(PATH, usecols=cols)
df.columns = df.columns.str.lower()
labels = {k.lower(): v for k, v in meta2.variable_value_labels.items()}
print(df.shape)

In [ ]:
N = len(df)
idx = np.arange(N)

def A(stem):
    """(N, 17) array; column w = wave w (column 0 unused). Missing waves = NaN."""
    out = np.full((N, 17), np.nan)
    for w in W:
        n = vname(stem, w)
        if n in df:
            out[:, w] = df[n].to_numpy(dtype=float)
    return out

def at(arr, t):
    """Value of arr at wave t for each person (NaN if t < 1)."""
    out = np.full(N, np.nan)
    ok = (t >= 1) & (t <= 16)
    out[ok] = arr[idx[ok], t[ok]]
    return out

In [ ]:
pen, work = A('peninc'), A('work')

onset = np.full(N, -1)
for w in range(16, 1, -1):
    onset[pen[:, w] == 1] = w

prior0 = np.zeros(N, bool)
for w in range(2, 16):
    prior0 |= (pen[:, w] == 0) & (w < onset)

nxt = np.where((onset >= 1) & (onset <= 15), onset + 1, -1)
two = at(pen, nxt) == 1
t0 = np.where(two, onset, -1)

t1 = np.full(N, -1)
for w in range(1, 16):
    t1[(work[:, w] == 1) & (w < onset)] = w

t0n = np.where(t0 >= 1, t0 + 1, -1)

In [ ]:
ptyp_lab = labels.get('r10ptyp1') or labels.get('r5ptyp1') or {}
print('PTYP labels:', ptyp_lab)

DB_CODES = [1]
DC_CODES = [2]

penct_t1 = at(A('penct'), t1)
types_t1 = np.column_stack([at(A(f'ptyp{k}'), t1) for k in range(1, 5)])
ptyp_t1 = types_t1[:, 0]

def all_in(codes):
    """현 직장의 모든 플랜(1..penct)이 주어진 유형 코드에 속하는가"""
    ok = penct_t1 >= 1
    for k in range(4):
        has_plan_k = penct_t1 >= k + 1
        ok &= ~has_plan_k | np.isin(types_t1[:, k], codes)
    return ok

is_db = (penct_t1 == 1) & np.isin(ptyp_t1, DB_CODES)   # DB: only one plan
is_dc = all_in(DC_CODES)                                # DC: all in DC accounts

# robustness
is_db_multi  = all_in(DB_CODES)                              # multiple DB
is_dc_single = (penct_t1 == 1) & np.isin(ptyp_t1, DC_CODES)  # one DC plan

print('Overlap DB & DC:', (is_db & is_dc).sum())   # must be zero

# DC account balance
dcbal_mat = np.column_stack([at(A(f'dcbal{k}'), t1) for k in range(1, 5)])
dcbal_t1 = np.nansum(dcbal_mat, axis=1)
dcbal_t1[np.isnan(dcbal_mat[:, 0])] = np.nan

PTYP labels: {1: '1.db', 2: '2.dc', 3: '3.db+dc', 'd': '.d=dk', 'j': '.j=web interview, missing', 'm': '.m=oth missing', 'q': '.q=not asked this wv', 'r': '.r=rf', 's': '.s=no pen info', 'u': '.u=unmarried', 'v': '.v=spouse nr', 'z': '.z=no pen based on penct'}
Overlap DB & DC: 0


In [ ]:
# ── Job history verification ─────────────────────────────────────────────
# t1 is the last wave with work == 1 before onset, but that alone does not
# guarantee that the t1 job is the job that pays the pension at t0:
#   (a) a missed interview between t1 and t0 could hide another job,
#   (b) work could be missing (not 0) in the waves in between,
#   (c) the t1 job must have usable, plausible tenure (YOS for the DB formula),
#   (d) if R reports a NEW employer since the previous interview, tenure on
#       the t1 job cannot exceed the time elapsed since that interview
#       (flag only, used as a robustness check). The same-employer direction
#       is not informative: RAND tenure is built from the job start year, so it
#       moves one-for-one with time whenever samemp == 1.
inw, samemp, jcten, age = A('inw'), A('samemp'), A('jcten'), A('agey_e')

# (a) interviewed in every wave in (t1, t0]
# (b) observed NOT working (work == 0, not missing) in every wave in (t1, t0)
inw_between  = np.ones(N, bool)
nowork_between = np.ones(N, bool)
for w in W:
    in_gap = (t1 >= 1) & (t0 >= 1) & (w > t1)
    inw_between    &= ~(in_gap & (w <= t0)) | (inw[:, w] == 1)
    nowork_between &= ~(in_gap & (w < t0))  | (work[:, w] == 0)

# (c) tenure on the t1 job observed, positive and below (age - 14)
ten_t1  = at(jcten, t1)
age_t1  = at(age, t1)
ten_ok  = (ten_t1 > 0) & (ten_t1 <= age_t1 - 14)

# (d) new employer since previous interview => tenure <= elapsed time + 1 year
samemp_t1 = at(samemp, t1)
prev = np.full(N, -1)                                  # last interviewed wave before t1
for w in W:
    prev[(inw[:, w] == 1) & (w < t1)] = w
elapsed = age_t1 - at(age, prev)                       # years since previous interview
comparable = (samemp_t1 == 0) & ~np.isnan(ten_t1) & ~np.isnan(elapsed)
ten_consistent = ~comparable | (ten_t1 <= elapsed + 1)          # tolerance: 1 year

job_ok = inw_between & nowork_between & ten_ok

print('Same employer as previous interview at t1:',
      pd.Series(samemp_t1).value_counts(dropna=False).to_dict())
print('New employer at t1, elapsed time observed :', comparable.sum())
print('  of which tenure > elapsed + 1 (flagged) :', (comparable & ~ten_consistent).sum())

In [ ]:
# CPI-U, all items, U.S. city average (FRED series CPIAUCSL, seasonally adjusted, 1982-84 = 100),
# annual average of the 12 monthly values. Source file: reference/CPIAUCSL.csv
CPI = {1990: 130.658, 1991: 136.167, 1992: 140.308, 1993: 144.475, 1994: 148.225,
       1995: 152.383, 1996: 156.858, 1997: 160.525, 1998: 163.008, 1999: 166.583,
       2000: 172.192, 2001: 177.042, 2002: 179.867, 2003: 184.000, 2004: 188.908,
       2005: 195.267, 2006: 201.558, 2007: 207.344, 2008: 215.254, 2009: 214.565,
       2010: 218.076, 2011: 224.923, 2012: 229.586, 2013: 232.952, 2014: 236.715,
       2015: 237.002, 2016: 240.005, 2017: 245.121, 2018: 251.100, 2019: 255.653,
       2020: 258.856, 2021: 270.973, 2022: 292.626, 2023: 304.703}
BASE_YEAR = 2022
_cpi = np.full(2100, np.nan)
for y, v in CPI.items():
    _cpi[y] = v

def to_real(year):
    """Multiplier converting nominal dollars of `year` into BASE_YEAR dollars."""
    year = np.asarray(year, float)
    out = np.full(year.shape, np.nan)
    ok = np.isfinite(year)
    out[ok] = CPI[BASE_YEAR] / _cpi[year[ok].astype(int)]
    return out


# ── Outcome: log real monthly pension, mean of t0 and t0+1 ──────────────
# RAND income amounts are annual and refer to the calendar year before the
# interview, so each wave's pension is deflated with CPI of (interview year - 1).
nominal_year = 1990 + 2 * np.arange(17)                # wave w -> 1990 + 2w
iwy = A('iwendy')
iwy = np.where(np.isnan(iwy), nominal_year[None, :], iwy)
print('Interview year from RAND (iwendy):', 'iwendy' in ''.join(df.columns))

pen_m_real = A('ipen') * to_real(iwy - 1) / 12        # (N, 17) real monthly pension
pen_m_t0, pen_m_t0n = at(pen_m_real, t0), at(pen_m_real, t0n)
pen_m = (pen_m_t0 + pen_m_t0n) / 2
ln_pen = np.where(pen_m > 0, np.log(np.where(pen_m > 0, pen_m, 1)), np.nan)

TRIM = 0.01                                             # drop top/bottom 1% within plan type
COHORTS = [3, 4, 5]                                     # HRS, War Babies, Early Baby Boomers

In [ ]:
ipen = A('ipen')
y0, y1 = at(ipen, t0), at(ipen, t0n)
pos = (y0 > 0) & (y1 > 0)

rows, m = [], np.ones(N, bool)
def step(name, cond):
    global m
    m = m & cond
    rows.append((name, m.sum(), (m & is_db).sum(), (m & is_dc).sum()))

step('0. All respondents', np.ones(N, bool))
step('1. Born 1931-1953, HRS/WB/EBB cohorts',
     df['rabyear'].between(1931, 1953).to_numpy() & df['hacohort'].isin(COHORTS).to_numpy())
step('2. Pension onset observed after a non-receipt wave', (onset >= 1) & prior0)
step('3. Receipt in 2 consecutive waves (t0 defined)', t0 >= 1)
step('4. Exactly one pension at t0', at(A('peni_n'), t0) == 1)
step('5. Working wave before t0 (t1 defined)', t1 >= 1)
step('6. Not self-employed at t1', at(A('slfemp'), t1) != 1)
step('7. Pension on t1 job', at(A('jcpen'), t1) == 1)
m_before_job = m.copy()                                # sample before job-history checks
step('8. Interviewed in every wave between t1 and t0', inw_between)
step('9. Not working in every wave between t1 and t0', nowork_between)
step('10. Tenure on t1 job observed and plausible', ten_ok)
m_before_plan = m.copy()                               # robustness check (plan definitions)
step('11. DB: single DB plan / DC: all plans DC', is_db | is_dc)
step('12. Positive pension income at t0 and t0+1', pos)
trim_ok = np.zeros(N, bool)
for g in (is_db, is_dc):
    s = m & g & ~np.isnan(ln_pen)
    if not s.any():
        continue
    lo, hi = np.quantile(ln_pen[s], [TRIM, 1 - TRIM])
    trim_ok |= s & (ln_pen >= lo) & (ln_pen <= hi)
    print(f'Trim bounds (monthly real $): {np.exp(lo):,.0f} - {np.exp(hi):,.0f}')
step('13. Outcome within 1st-99th percentile (by plan type)', trim_ok)

att = pd.DataFrame(rows, columns=['Step', 'N', 'of which DB', 'of which DC'])
print(att.to_string(index=False))

final = m.copy()
print('\nDC with t0 >= wave 5 (IRA withdrawal available):', (final & is_dc & (t0 >= 5)).sum())
print('DB with earnings at t1 non-missing:', (final & is_db & ~np.isnan(at(A('iearn'), t1))).sum())
print('DC with balance at t1 non-missing (summed):', (final & is_dc & ~np.isnan(dcbal_t1)).sum())

print('\nJob-history checks, each applied alone (steps 0-7, 11-12 imposed, no trimming):')
base = m_before_job & (is_db | is_dc) & pos
for name, c in [('interviewed between t1 and t0', inw_between),
                ('not working between t1 and t0', nowork_between),
                ('tenure observed and plausible ', ten_ok),
                ('tenure vs. new employer (flag)', ten_consistent)]:
    print(f'  {name}: fail DB {(base & is_db & ~c).sum():4d} / DC {(base & is_dc & ~c).sum():4d}')

print('\nRobustness (same steps, alternative definitions):')
print('DB, multiple DB plans allowed  :', (m_before_plan & is_db_multi & pos).sum())
print('DC, single DC plan only        :', (m_before_plan & is_dc_single & pos).sum())
print('DB, tenure-consistent only     :', (final & is_db & ten_consistent).sum())
print('DC, tenure-consistent only     :', (final & is_dc & ten_consistent).sum())
print('DB, without job-history checks :', (m_before_job & is_db & pos).sum())
print('DC, without job-history checks :', (m_before_job & is_dc & pos).sum())

In [ ]:
keep = m_before_job & pos & (is_db | is_dc | is_db_multi | is_dc_single)

out = pd.DataFrame({
    'hhidpn':        df['hhidpn'].to_numpy(),
    'rabyear':       df['rabyear'].to_numpy(),
    'hacohort':      df['hacohort'].to_numpy(),
    't0':            t0,
    't1':            t1,
    'gap_t0_t1':     np.where((t0 >= 1) & (t1 >= 1), t0 - t1, np.nan),
    'penct_t1':      penct_t1,
    'ptyp1_t1':      ptyp_t1,
    'dcbal_t1':      dcbal_t1,
    'age_t1':        age_t1,
    'jcten_t1':      ten_t1,
    'samemp_t1':     samemp_t1,
    'ipen_t0':       y0,
    'ipen_t0n':      y1,
    # outcome (BASE_YEAR dollars)
    'iwy_t0':        at(iwy, t0),
    'pen_m_t0':      pen_m_t0,
    'pen_m_t0n':     pen_m_t0n,
    'pen_m':         pen_m,
    'ln_pen':        ln_pen,
    'ifpen_t0':      at(A('ifpen'), t0),     # RAND imputation flag, R pension
    'ifpen_t0n':     at(A('ifpen'), t0n),
    # sample defining
    'is_db':         is_db,
    'is_dc':         is_dc,
    'is_db_multi':   is_db_multi,
    'is_dc_single':  is_dc_single,
    'dc_ira_ok':     t0 >= 5,
    # job history verification
    'inw_between':   inw_between,
    'nowork_between': nowork_between,
    'ten_ok':        ten_ok,
    'ten_consistent': ten_consistent,
    'job_ok':        job_ok,
    'in_main':       final,
})[keep].reset_index(drop=True)

# main sample = all steps passed; other candidates are kept for robustness
out['type'] = np.select([out['in_main'] & out['is_db'], out['in_main'] & out['is_dc']],
                        ['DB', 'DC'], default='')

# main sample distribution
main = out[out['type'] != '']
print('Main sample:', main['type'].value_counts().to_dict())
print('\nClaiming wave (t0) by type:')
print(pd.crosstab(main['t0'], main['type']))
print('\nCohort by type:')
print(pd.crosstab(main['hacohort'], main['type']))
print('\nGap between t1 and t0 (waves):')
print(main.groupby('type')['gap_t0_t1'].describe())
print(f'\nReal monthly pension ({BASE_YEAR} $):')
print(main.groupby('type')['pen_m'].describe().round(0))
print('\nRAND imputation flag of the pension amount (r#ifpen), t0:', labels.get('r10ifpen', ''))
print(pd.crosstab(main['ifpen_t0'], main['type']))

SAVE = '/content/drive/MyDrive/HRS/sample_ids.parquet'
out.to_parquet(SAVE, index=False)
print('\nSaved', len(out), 'rows to', SAVE)